**Focus of the document**


This document focuses on the utilization of basic ML methods that are taught in the lecture of the course DSA210. The ML methods are somewhat considered to be basic due to nature of the dataset, and the course content.


Possible ideas;


- can daily entertainment behaviour signal about academic pressure
- Regression for the sequential data especially spotify is the best based on the EDA we have
- Predicting maybe whether a day has high activity after 21.30.
 




In [ ]:
# The question focused : Can daily entertainment behaviour a signal about academic pressure.

# As a binary classification.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from pathlib import Path

In [5]:
#----------------------DATASET PREPARATION-----------------------#
# Adding the combined daily dataset to the machine learning dataset.

project_root = Path.cwd()
if project_root.name == "MachineLearning":
    project_root = project_root.parent


youtube_path = project_root / "data_github/youtube_public/youtube_activity_public.csv"
spotify_path = project_root / "data_github/spotify_public/spotify_streaming_public.csv"
netflix_path = project_root / "data_github/netflix_public/netflix_viewing_public.csv"
prime_path = project_root / "data_github/prime_video_public/prime_video_watch_history_public.csv"
calendar_path = project_root / "data_github/academical_calendar/academicCalendar.jsonl"


In [6]:
#--------------LOADING THE DATASETS-----------------------#
youtube_data = pd.read_csv(youtube_path)
spotify_data = pd.read_csv(spotify_path)
netflix_data = pd.read_csv(netflix_path)
prime_data = pd.read_csv(prime_path)
calendar_data = pd.read_json(calendar_path, lines=True)

print("Loaded datasets:")
print("YouTube:", youtube_data.shape)
print("Spotify:", spotify_data.shape)
print("Netflix:", netflix_data.shape)
print("Prime Video:", prime_data.shape)
print("Academic calendar:", calendar_data.shape)

Loaded datasets:
YouTube: (34317, 16)
Spotify: (178202, 16)
Netflix: (2493, 11)
Prime Video: (719, 17)
Academic calendar: (15, 23)


In [8]:

# Having a common date that we would build the foundation of the ML.

# For each of the data set making sure that we use a common date format and a common date column name. We will use "fine_date" as the common date column name.
for data in [youtube_data, spotify_data, netflix_data, prime_data]:
    data["fine_date"] = pd.to_datetime(data["fine_date"])



# YouTube and Spotify have exact timestamps, so they can be used for after-21:30 variables.
youtube_data["fine_time_start_dt"] = pd.to_datetime(
    youtube_data["fine_time_start"],
    errors="coerce",
    utc=True
).dt.tz_convert("Europe/Istanbul")

spotify_data["fine_time_start_dt"] = pd.to_datetime(
    spotify_data["fine_time_start"],
    errors="coerce",
    utc=True,
    format="mixed"
).dt.tz_convert("Europe/Istanbul")


# Spotify has different timestamps formats, I was getting 5404 rows rthat had invalid timestamps. I will try to parse them with a different format.
print("Invalid Spotify timestamps:", spotify_data["fine_time_start_dt"].isna().sum())


# Since we are interested in the late-evening / night activity, we will create a binary variable that indicates whether the activity started after 21:30 or not. This will be our target variable for the machine learning model.
# After 21:30 until 04:59 is treated as late-evening / night activity.
youtube_data["is_after_2130"] = (
    (youtube_data["fine_time_start_dt"].dt.hour > 21) |
    (
        (youtube_data["fine_time_start_dt"].dt.hour == 21) &
        (youtube_data["fine_time_start_dt"].dt.minute >= 30)
    ) |
    (youtube_data["fine_time_start_dt"].dt.hour < 5)
)

# Same for Spotify.

spotify_data["is_after_2130"] = (
    (spotify_data["fine_time_start_dt"].dt.hour > 21) |
    (
        (spotify_data["fine_time_start_dt"].dt.hour == 21) &
        (spotify_data["fine_time_start_dt"].dt.minute >= 30)
    ) |
    (spotify_data["fine_time_start_dt"].dt.hour < 5)
)

#Checkcing before the ML so no errors in the timestamps.

print("Invalid YouTube timestamps:", youtube_data["fine_time_start_dt"].isna().sum())
print("Invalid Spotify timestamps:", spotify_data["fine_time_start_dt"].isna().sum())


Invalid Spotify timestamps: 0
Invalid YouTube timestamps: 0
Invalid Spotify timestamps: 0


In [11]:


# Adding a common date range for all datasets to make sure that we are comparing the same time periods. We will create a new dataframe that contains all the dates in the common date range and then merge it with each of the datasets.
common_start_date = max(
    youtube_data["fine_date"].min(),
    spotify_data["fine_date"].min(),
    netflix_data["fine_date"].min(),
    prime_data["fine_date"].min()
)


# Adding a common end date as well, to make sure that we are comparing the same time periods. We will create a new dataframe that contains all the dates in the common date range and then merge it with each of the datasets.
common_end_date = min(
    youtube_data["fine_date"].max(),
    spotify_data["fine_date"].max(),
    netflix_data["fine_date"].max(),
    prime_data["fine_date"].max()
)

all_dates = pd.DataFrame({
    "fine_date": pd.date_range(common_start_date, common_end_date, freq="D")})



calendar_rows = []


# Assigning the classes based on the academic calendar.
# We will create a new dataframe that contains all the dates in the common date range and then merge it with each of the datasets. We will also create a new column that indicates whether the date is in the ordinary term, final exam period or summer work period.
for _, row in calendar_data.iterrows():
    term_start = pd.to_datetime(row["term_start_date"])
    final_start = pd.to_datetime(row["final_exam_start_date"])
    final_end = pd.to_datetime(row["final_exam_end_date"])

    for date_value in pd.date_range(term_start, final_end, freq="D"):
        if final_start <= date_value <= final_end:
            academic_period = "final_exam"
        else:
            academic_period = "ordinary_term"

        if row["term"] == "summer":
            analysis_period = "summer_work_period"
        else:
            analysis_period = academic_period

        calendar_rows.append({
            "fine_date": date_value,
            "analysis_period": analysis_period
        })

calendar_daily = pd.DataFrame(calendar_rows).drop_duplicates("fine_date", keep="first")

print("Common date range:")
print(common_start_date, "to", common_end_date)
print("Number of dates:", len(all_dates))


Common date range:
2022-04-17 00:00:00 to 2026-03-10 00:00:00
Number of dates: 1424


In [12]:
#-----------------------DATASET-----------------------#

